# Revisiting SASRec on Amazon Reviews 2023

**Team:** Alina Chuang, Frank Bartkus, Kyle Nguyen, Matthew Mikhin

We re-run the experiments of SASRec (Kang & McAuley, ICDM 2018) [1] on the 2023 release of the
Amazon review data [2]. Our main question is whether the paper's sampled evaluation and a full
ranking over all items put the methods in the same order.

This notebook is the whole project and runs from top to bottom.

- **The rules everyone follows** are in the TODO box of the member who builds that part, so
  that every method is trained and measured in the same way. Section 2 lists where each is.
- **Every other section starts with a TODO list for its owner.** Delete a TODO item when it is
  done; delete the whole TODO box when the section is finished.

**Contents**

0. About: how to run, who does what, contribution statement, use of AI tools
1. Problem definition
2. Settings
3. Data
4. Evaluation
5. Methods
6. Training and tuning
7. Results
8. Sampled evaluation vs full ranking: our main question
9. Ablations: which parts of SASRec matter? (if time allows)
10. Demo
11. Discussion: observations, limitations, future work
12. References

## 0. About this notebook

### How to run it

```bash
uv sync                  # once: installs the Python packages from uv.lock into .venv
uv run jupyter lab       # or open the notebook in VS Code and pick the .venv kernel
```

Then run all cells from the top. The first run downloads the data (about 14 MB, Section 3.1). With `RUN_EXPERIMENTS = False` (Section 2) the trained
results are read from `results/` and the notebook runs in minutes; with `True` every model is
trained again, which takes hours.

How we work on this one file together (git, branches, outputs) is in `docs/team_spec.md`.

### Who does what, and in which order

Hand-in: **Monday, November 30, 23:55**. Presentations: December 1 and 3.

**What comes first.** The teacher's feedback on our proposal says the scope is large and
suggests this order, which we follow:

1. **Core:** SASRec, PopRec and BPR, each under both protocols (sampled and full ranking). This
   answers our main question (RQ1, Section 8).
2. **If time allows, in this order:** FMC and FPMC (Section 5.2), then the ablations
   (Section 9). Train these only after the core results are in `results/`.

| # | Task | Who | Needs first | Target |
|---|---|---|---|---|
| 0 | The rules every method follows, written into each owner's section (list in Section 2) | Alina | — | **done** Wed Nov 18 |
| 1 | Choose the category: `Industrial_and_Scientific` (Section 3.2) | Alina | — | **done** Mon Nov 23 |
| 2 | Data loading and splitting (Section 3.1) | Frank | — | **done** Tue Nov 24 |
| 3 | Negatives, metrics, `recommend` (Section 4) | Frank | 2 |  **done** Wed Nov 25 |
| 4 | PopRec (Section 5.1) | Frank | 3 |  **done** Wed Nov 25 |
| 5 | SASRec and the training loop, on fake data (Sections 5.3, 6) | Alina | — | Tue Nov 24 |
| 6 | Problem definition, related work, references (Sections 1, 12) | Kyle | — | Wed Nov 25 |
| 7 | Data checks and EDA (Section 3.2) | Matthew | 2 | Wed Nov 25 |
| 8 | BPR, tuned on validation, test with 3 seeds (Section 5.2) | Kyle | 3, 5 | Fri Nov 27 |
| 9 | SASRec, tuned on validation, test with 3 seeds, model saved for the demo | Alina | 3, 5 | Fri Nov 27 |
| 10 | Results table and RQ1, sampled vs full (Sections 7, 8) | Frank | 4, 8, 9 | Sat Nov 28 |
| 11 | Results by history length (Section 8) | Matthew | 8, 9 | Sat Nov 28 |
| 12 | Answer RQ2 (Section 7) | Kyle | 10 | Sat Nov 28 |
| 13 | Figures (Sections 3.2, 7, 8) | Matthew | 10 | Sat Nov 28 |
| 14 | Demo cell and demo video (Section 10) | Matthew | 4, 9 | Sun Nov 29 |
| 15 | Discussion, contribution statement, AI note (Sections 0, 11) | everyone | 10 | Sun Nov 29 |
| 16 | Final run with outputs; hand in the notebook and the video | Matthew | everything | Mon Nov 30 |
| — | If time allows: FMC and FPMC (Kyle), then the ablations (Alina) | Kyle, Alina | 8, 9 | Sat Nov 28 |

Steps 2 and 3 block everyone else. Thursday, November 26 is Thanksgiving: if you are away
around then, tell the team now.

### Contribution statement

> **TODO (everyone, at the end).** One row per member, agreed by all four.

| Member | Contribution |
|---|---|
| Alina Chuang | |
| Frank Bartkus | |
| Kyle Nguyen | |
| Matthew Mikhin | |

### Use of generative AI tools

> **TODO (everyone).** The course requires a short note on which AI tools each of us used and
> for what. Each member writes their own row.

| Member | Tool | Used for |
|---|---|---|
| Alina Chuang | | |
| Frank Bartkus | | |
| Kyle Nguyen | | |
| Matthew Mikhin | | |

## 1. Problem definition

> **TODO (Kyle)**
> - [ ] What a sequential recommender does, with a small example of one user's history and
>   the next item.
> - [ ] SASRec in one paragraph. Alina sends you her method paragraph (Section 5.3).
> - [ ] Why re-test on the 2023 data: what changed since the paper's data.
> - [ ] The three research questions (`README.md`). RQ1, sampled evaluation vs full ranking, is
>   our main question: the teacher asked us to keep it at the center.
> - [ ] The order of PopRec, BPR, FMC, FPMC and SASRec in the paper's Table III, for Beauty and
>   for Games. Our data differs from the paper's, so we compare orders, not exact numbers
>   (teacher's feedback). RQ2 (Section 7) uses this.
> - [ ] Related work, each paper cited in the text as [n] (list in Section 12): SASRec [1],
>   Amazon Reviews 2023 [2], BPR [3], FMC and FPMC [4], sampled metrics [5], why baselines
>   must be tuned [6].

## 2. Settings (Alina)

SASRec is compared with PopRec and BPR, and with FMC and FPMC if time allows. That comparison
only means something if every method gets the same data, the same test cases, the same negatives, the same metric and
the same tuning. A difference in any of these would show up as a difference between methods.
So the team fixes these rules once, and every section follows them.

Alina drafted the rules. They are final once every member has read them and nobody objects
(by Thursday, November 19). Each rule is written in the TODO box of the member who builds that
part, because that member explains it in the report and in the presentation:

| Rules for | Section | Owner |
|---|---|---|
| reading the data and splitting it | 3.1 | Frank |
| negatives, ranks, metrics | 4 | Frank |
| what every model looks like | 5 | Matthew |
| training, tuning, early stopping | 6 | Alina |
| result files and method names | 7 | Frank |
| names of the ablations | 9 | Alina |

If you think a rule should change, tell the team first; do not change it on your own.

This section holds only the fixed numbers that several sections use, and `set_seed`. They come
first because the code of every later section uses them.

In [1]:
import os
import random
import urllib.request

import pandas as pd
import torch

# The Amazon Reviews 2023 category we use; why this one: Section 3.2.
CATEGORY = "Industrial_and_Scientific"

# True: train every model again (hours). False: read the saved results from results/.
RUN_EXPERIMENTS = False

# Evaluation, as in the paper (Section IV-D)
EVAL_SEED = 0  # one fixed set of 100 negatives, the same for every method
N_NEGATIVES = 100  # each answer is ranked against 100 items the user never touched
K = 10  # we report Hit@10 and NDCG@10

# Training
TRAIN_SEEDS = [1, 2, 3]  # every trained model runs 3 times; we report mean and std
PATIENCE = 10  # stop after 10 epochs without a better validation NDCG@10 (paper: 20)
MAX_EPOCHS = 100  # never train longer than this, so every run ends
MAX_LEN = 50  # SASRec reads at most the last 50 items (paper IV-C)

Add the imports you need to the cell above, so all imports are in one place.

A model uses random numbers for its starting weights, for picking negative items and for the
order of the batches. If two runs with the same seed gave different results, our mean over 3
seeds would mean nothing. `set_seed` puts every random number generator we use back to the
same starting point. The training loop calls it before it creates a model.

In [2]:
def set_seed(seed):
    """Fix every random number generator we use, so a run can be repeated exactly."""
    random.seed(seed)  # Python's own generator, for example random.sample
    torch.manual_seed(seed)  # torch on the CPU, and on the GPU too (CUDA or Apple MPS)


# Check: the same seed gives the same numbers.
set_seed(1)
first = torch.rand(3)
set_seed(1)
second = torch.rand(3)
print(first)
print(second)
print("same:", torch.equal(first, second))

tensor([0.7576, 0.2793, 0.4031])
tensor([0.7576, 0.2793, 0.4031])
same: True


## 3. Data

### 3.1 Source and loading (Frank)

**Source.** We use the official 5-core files of Amazon Reviews 2023 [2] for `CATEGORY`, which
the dataset's authors already split with the leave-one-out strategy (`last_out_w_his`): for
every user, the last item is the test item, the second-to-last item is the validation item,
and everything before is for training. The paper uses the same split (Section IV-A). The first
run downloads the three files (train, valid, test; about 14 MB) from the
[official site](https://amazon-reviews-2023.github.io/data_processing/5core.html) into
`data/<CATEGORY>/`; later runs read them from there. Every review counts as one interaction and
the star rating is not used, because the paper treats reviews as implicit feedback.

**How we build each user's sequence, and why:**

| Step | What we do | Why |
|---|---|---|
| Order | the user's train rows sorted by `timestamp` with a **stable** sort, then the valid item, then the test item | rows with the same timestamp keep their file order, so every computer gets the same sequence |
| User numbers | 0, 1, 2, … in order of first appearance in the train file | the same numbers on every computer; a Python `set` would give a different order on every run |
| Item numbers | 0, 1, 2, … in order of first appearance, reading train, then valid, then test | same reason; an item seen only in valid or test still gets a number |
| Repeated user–item pairs | kept as they are in the files (Matthew counts them in Section 3.2) | the official files are the protocol; changing them would make our data differ from everyone else's |
| Validation | answer = second-to-last item; input = all items before it | leave-one-out (IV-A) |
| Test | answer = last item; input = all items before it, **including the validation item** | the paper does the same (IV-A) |
| The official `history` column | not read | we build every history from our own sequences, so all sections use exactly the same ones |
| Check | the notebook stops if a user does not have exactly one valid row and one test row | the split above needs exactly one of each |

**The names every later section uses:**

| Name | What it is |
|---|---|
| `sequences` | `sequences[u]`: item numbers of user `u`, oldest first: train items, valid item, test item |
| `user_ids`, `item_ids` | the original ids: `user_ids[u]` is the `user_id` of user `u`, `item_ids[j]` the `parent_asin` of item `j` |
| `n_users`, `n_items` | `len(user_ids)`, `len(item_ids)` |
| `train_seqs[u]` | `sequences[u][:-2]`: what models learn from, and the validation input |
| `valid_answers[u]` | `sequences[u][-2]` |
| `test_histories[u]` | `sequences[u][:-1]` |
| `test_answers[u]` | `sequences[u][-1]` |

Item number `j` is column `j` of every score matrix.

In [3]:
# where the official leave-one-out files of every category are
DATA_URL = (
    "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/benchmark/"
    "5core/last_out_w_his/"
)


# reads one official file, and downloads it into data/<category>/ the first time
def read_split(category, split):
    name = category + "." + split + ".csv.gz"
    path = os.path.join("data", category, name)
    if not os.path.exists(path):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        urllib.request.urlretrieve(DATA_URL + name, path)
    # only the three columns we use: no star rating, no official history
    return pd.read_csv(path, usecols=["user_id", "parent_asin", "timestamp"])


# stops the notebook unless every user has exactly one valid row and one test row
def check_one_row_per_user(train_df, valid_df, test_df):
    train_users = set(train_df["user_id"])
    for name, df in [("valid", valid_df), ("test", test_df)]:
        # how many rows every user has in this file
        counts = df["user_id"].value_counts()
        missing = len(train_users - set(counts.index))
        if missing or (counts != 1).any():
            raise ValueError(
                f"every user needs exactly one {name} row: {missing} users have none, "
                f"{(counts > 1).sum()} users have more than one"
            )


# returns non duplicate values in list in order of earliest occurance
def unique(m_list):
    m_list = list(m_list)
    # pairs every value with its position, then keeps the first position of each value
    positions = pd.DataFrame(zip(m_list, range(len(m_list)), strict=True))
    return positions.groupby(0).min().sort_values(by=1).index.to_list()


def load_data(category):
    # the train, valid and test sets from the data files
    df_list = [read_split(category, t) for t in ["train", "valid", "test"]]
    train_df, valid_df, test_df = df_list
    check_one_row_per_user(train_df, valid_df, test_df)
    # the unique user and item ids sorted by earliest occurrence
    user_ids = unique(train_df["user_id"])
    item_ids = unique([d for df in df_list for d in df["parent_asin"]])
    # series to convert item ids to its respective indices
    item_id_to_index = pd.Series(range(len(item_ids)), index=item_ids)
    # the user and item ids sorted by timestamp for each file in the datafiles
    sorted_dfs = [
        df.sort_values(by="timestamp", kind="stable").drop(columns="timestamp")
        for df in df_list
    ]
    # groups the item ids by the user ids
    sorted_dfs = [df.groupby("user_id").agg(list)["parent_asin"] for df in sorted_dfs]
    # converts the item ids into their indices
    sorted_dfs = [
        pd.Series(
            [[item_id_to_index[i] for i in items] for items in df], index=df.index
        )
        for df in sorted_dfs
    ]
    # the train items, the valid item and the test item of every user, in one list
    sequences = [[int(d) for df in sorted_dfs for d in df[u]] for u in user_ids]
    # returns the sequences, the user ids, and the item ids
    return sequences, user_ids, item_ids

In [4]:
sequences, user_ids, item_ids = load_data(CATEGORY)
n_users, n_items = len(user_ids), len(item_ids)
# should match the official table: 51.0K users, 25.8K items, 412.9K interactions
n_interactions = sum(len(s) for s in sequences)
print(f"users: {n_users}  items: {n_items}  interactions: {n_interactions}")

users: 50985  items: 25848  interactions: 412947


In [5]:
# splits every user's sequence into the four lists the later sections use
def leave_one_out(sequences):
    for s in sequences:
        # every user needs at least one train item, the valid item and the test item
        if len(s) < 3:
            raise ValueError("every sequence must have at least 3 items")
    # train items: everything except the last two
    train_seqs = [s[:-2] for s in sequences]
    # the valid answer: the second-to-last item
    valid_answers = [s[-2] for s in sequences]
    # the test input: everything before the last item, so it includes the valid item
    test_histories = [s[:-1] for s in sequences]
    # the test answer: the last item
    test_answers = [s[-1] for s in sequences]
    return train_seqs, valid_answers, test_histories, test_answers

In [6]:
train_seqs, valid_answers, test_histories, test_answers = leave_one_out(sequences)

### 3.2 Data checks and exploratory analysis (Matthew)

> **TODO (Matthew)**
> - [ ] The category is `Industrial_and_Scientific`, chosen on November 23 from the official
>   5-core table (Amazon Reviews 2023 page, "Pure IDs (5-Core)"), after the teacher's
>   feedback asked for a small category. Explain the choice here in your own words, because
>   you present the data. The facts behind it:
>
>   | | Paper: Amazon Beauty (Table II) | Paper: Amazon Games (Table II) | Ours: `Industrial_and_Scientific` 2023, 5-core |
>   |---|---|---|---|
>   | users | 52,024 | 31,013 | 51.0K |
>   | items | 57,289 | 23,715 | 25.8K |
>   | interactions | 0.4M | 0.3M | 412.9K |
>
>   - Small, as the teacher advised: about the size of the paper's datasets, so every model
>     can be trained several times on our laptops (Section 6).
>   - The three `last_out_w_his` files are 7.1 MB, 3.5 MB and 3.7 MB, so the data can live in
>     this repository.
>   - It is not the same kind of product as the paper's Beauty and Games. That is fine,
>     because we compare the order of the methods, not the exact numbers (teacher's
>     feedback).
>   - Not `Video_Games`, our first choice: 94.8K users and 814.6K interactions, about twice
>     the work. Not `Beauty_and_Personal_Care`: 729.6K users. Not `All_Beauty`: only 253
>     users after 5-core filtering.
> - [ ] The size after 5-core filtering, as in the paper's Table II (users, items, average
>   interactions per user and per item, interactions), next to the paper's numbers. The
>   teacher asked us to report it.
> - [ ] One figure: interactions per user, log scale on both axes.
> - [ ] The two checks the proposal promises: how many repeated user–item pairs, and how many
>   timestamp ties within one user's items.

## 4. Evaluation (Frank)

> **TODO (Frank)** The team agreed on the rules in the tables below, so that every method is
> measured in exactly the same way. Build the evaluation by following them exactly. Then
> replace the tables with your own explanation: each rule and why, in your own words, because
> you present this part. If you think a rule should change, tell the team first.
>
> | Decision | Rule | Why |
> |---|---|---|
> | Negatives | for every user, separately for validation and for test: 100 items drawn uniformly, without repeats, from the items that appear **nowhere** in the user's sequence | the paper's sampled evaluation (IV-D); an item the user did interact with is not a fair negative |
> | Random generator for negatives | `random.Random(EVAL_SEED)`, users in index order, all validation negatives first, then all test negatives | the same negatives on every computer and for every method |
> | Candidate format | a tensor of shape `(n_users, 101)`: row `u` is user `u`, **column 0 is the answer**, columns 1 to 100 the negatives | the metric code finds the answer without a search |
> | Sampled rank `r` | 1 + the number of negatives whose score is **≥** the answer's score | ties count against the answer, so a model that gives every item the same score gets rank 101, not 1 |
> | Full-ranking rank `r` | 1 + the number of other items, not in the input history, whose score is **≥** the answer's score | the same tie rule over all items; items the user already has are left out, except the answer itself |
> | Hit@10 | 1 if `r ≤ 10`, else 0; averaged over users | paper IV-D |
> | NDCG@10 | `1 / log2(r + 1)` if `r ≤ 10`, else 0; averaged over users | paper IV-D; with one answer this is the whole formula |
>
> - [x] The negatives: `valid_candidates` and `test_candidates`, each of shape
>   `(n_users, 101)` with the answer in column 0. Make them once, here; every method is
>   measured on these same two tensors (teacher's feedback: same candidate lists for every
>   model).
> - [x] `evaluate_sampled(model, histories, candidates)` returns
>   `{"hit@10": ..., "ndcg@10": ..., "ranks": ...}`, where `ranks` holds every user's rank `r`
>   (Section 8 uses it). The training loop calls it after every epoch.
> - [x] `evaluate_full(model, histories, answers)`: the same, with the full-ranking rank.
> - [x] `recommend(model, users, histories, k)`: the top `k` items per case with the input
>   history removed, for the demo (Section 10). Hint: write `-inf` into a copy of the
>   scores, never into the tensor the model returned (PopRec can return its own stored scores).

In [ ]:
# set_seed(EVAL_SEED) reset both Python and PyTorch’s global randomness, so every run of
# the notebook uses the same 100 negatives for every user.
# keep negatives same for every run
eval_rng = random.Random(EVAL_SEED)
item_set = set(range(n_items))

In [ ]:
# Exclude all known interactions from negatives
valid_candidates = []
for sequence, answer in zip(
    sequences, valid_answers, strict=True
):  # pairs each answer with its user’s full sequence
    unseen_items = sorted(item_set - set(sequence))
    negatives = eval_rng.sample(unseen_items, N_NEGATIVES)
    valid_candidates.append([answer] + negatives)

test_candidates = []
for sequence, answer in zip(sequences, test_answers, strict=True):
    unseen_items = sorted(item_set - set(sequence))
    negatives = eval_rng.sample(unseen_items, N_NEGATIVES)
    test_candidates.append([answer] + negatives)

valid_candidates = torch.tensor(valid_candidates, dtype=torch.long)
test_candidates = torch.tensor(test_candidates, dtype=torch.long)

In [ ]:
class ConstantModel:  # gives every item the same score
    def __init__(self, n_items):
        self.n_items = n_items

    def score(self, users, histories):
        return torch.zeros(len(users), self.n_items)


class AnswerModel:  # FOR TESTING ONLY
    def __init__(self, n_items, answers):
        self.n_items = n_items
        self.answers = answers

    def score(self, users_, histories):
        s = torch.zeros(len(users_), self.n_items)
        rows = torch.arange(len(users_))
        s[rows, torch.tensor([self.answers[u] for u in users_])] = 1.0
        return s

In [10]:
def ranks_to_result(ranks):
    hit = torch.where(ranks <= K, 1, 0.0).mean()
    ndcg = torch.where(ranks <= K, 1 / torch.log2(ranks + 1), 0).mean()
    return {"hit@" + str(K): hit, "ndcg@" + str(K): ndcg, "ranks": ranks}

In [ ]:
# avoid full matrix; no need for gradients here
@torch.no_grad()
def evaluate_sampled(model, histories, candidates, batch_size=256):
    candidates = torch.as_tensor(candidates, dtype=torch.long)
    ranks = []
    for start in range(0, len(histories), batch_size):  # batch scoring
        end = min(start + batch_size, len(histories))
        score = model.score(range(start, end), histories[start:end])
        selected = score.gather(1, candidates[start:end])  # use tensor operations
        ranks.append(1 + (selected[:, 1:] >= selected[:, :1]).sum(dim=1))
    return ranks_to_result(torch.cat(ranks))

In [ ]:
# Full ranking takes one answer per user; candidate lists belong to sampled ranking.
@torch.no_grad()
def evaluate_full(model, histories, answers, batch_size=256):
    answers = torch.as_tensor(answers, dtype=torch.long)
    ranks = []
    for start in range(0, len(histories), batch_size):
        end = min(start + batch_size, len(histories))
        score = model.score(
            range(start, end), histories[start:end]
        ).clone()  # use clone to copy before masking
        rows = torch.arange(end - start)
        answer_list = score[rows, answers[start:end]]
        for row, history in enumerate(histories[start:end]):
            score[row, history] = -torch.inf  # exlude seen itgems
        score[rows, answers[start:end]] = -torch.inf
        ranks.append(1 + (score >= answer_list[:, None]).sum(dim=1))
    return ranks_to_result(torch.cat(ranks))

In [ ]:
# Batch recommendations to limit memory; inference needs no gradients.
@torch.no_grad()
def recommend(model, users, histories, k, batch_size=256):
    recommendations = []
    for start in range(0, len(users), batch_size):
        end = min(start + batch_size, len(users))
        score = model.score(
            users[start:end], histories[start:end]
        ).clone()  # again, copy b4 masking
        for row, history in enumerate(histories[start:end]):
            score[row, history] = -torch.inf  # exclude seen items
        recommendations.append(torch.topk(score, k, dim=1).indices)
    return torch.cat(recommendations)

In [14]:
const_model_ = ConstantModel(n_items)
ans_model_ = AnswerModel(n_items, valid_answers)

In [15]:
# 0, 0, all 101s
evaluate_sampled(const_model_, train_seqs, valid_candidates)

{'hit@10': tensor(0.),
 'ndcg@10': tensor(0.),
 'ranks': tensor([101., 101., 101.,  ..., 101., 101., 101.])}

In [16]:
# 0, 0, n_items - len(history) (25844, 25843, 25791,  ..., 25845, 25843, 25845)
evaluate_full(const_model_, train_seqs, valid_answers)

{'hit@10': tensor(0.),
 'ndcg@10': tensor(0.),
 'ranks': tensor([25844, 25843, 25791,  ..., 25845, 25843, 25845])}

In [17]:
recommend(const_model_, range(n_users), train_seqs, K)

tensor([[ 8,  9,  4,  ..., 13, 10,  6],
        [ 3,  9,  1,  ..., 14, 10,  2],
        [ 8,  4,  1,  ..., 66,  2,  6],
        ...,
        [ 8,  9,  4,  ...,  0,  2,  6],
        [ 8,  9,  4,  ...,  0,  2,  6],
        [ 8,  9,  4,  ...,  0,  2,  6]])

In [18]:
# 1, 1, all 1s
evaluate_sampled(ans_model_, train_seqs, valid_candidates)

{'hit@10': tensor(1.),
 'ndcg@10': tensor(1.),
 'ranks': tensor([1., 1., 1.,  ..., 1., 1., 1.])}

In [19]:
# 1, 1, all 1s
evaluate_full(ans_model_, train_seqs, valid_answers)

{'hit@10': tensor(1.),
 'ndcg@10': tensor(1.),
 'ranks': tensor([1, 1, 1,  ..., 1, 1, 1])}

In [20]:
recommend(ans_model_, range(n_users), train_seqs, K)

tensor([[ 4843,     4,     8,  ...,     5,    13,    10],
        [ 3506,     1,     3,  ...,     0,    14,    10],
        [ 2624,     1,     8,  ...,     5,    66,     2],
        ...,
        [ 9233,     4,     8,  ...,     5,     0,     2],
        [12300,     4,     8,  ...,     5,     0,     2],
        [21668,     4,     8,  ...,     5,     0,     2]])

## 5. Methods

> **TODO (Matthew)** The team agreed on the rules below, so that one training loop and one
> evaluation work for every method. Every model owner follows them. Replace them with your own
> explanation, because you present how the parts of the notebook fit together.
>
> Every method (PopRec, BPR, SASRec; FMC and FPMC if time allows) is a class with three methods:
>
> | Method | What it does |
> |---|---|
> | `__init__(self, n_users, n_items, ...)` | creates the model; settings such as `d` come after the first two arguments, with the paper's value as the default |
> | `train_one_epoch(self, train_seqs)` | learns from the training sequences once and returns the average loss as a `float` |
> | `score(self, users, histories)` | returns a tensor of shape `(len(users), n_items)`: a score for every item for every case; higher means recommend earlier |
>
> Case `r` is user `users[r]` with input history `histories[r]`. BPR and FPMC need the user;
> FMC and SASRec need the history; PopRec needs neither.
>
> Rules:
>
> 1. A model learns only from what `train_one_epoch` receives. Never read a global variable
>    such as `test_answers` or `sequences` inside a model.
> 2. Never call `set_seed` inside a model, and never create a random generator with a fixed
>    number inside a model. The training loop seeds everything before the model is created.
> 3. Every setting is an argument of `__init__`; no numbers hidden inside methods. Ablations
>    are settings too, for example `use_pe=False`.
> 4. `score` returns exactly `n_items` columns. A padding item (SASRec) is removed before
>    returning.
> 5. No metric code inside a model; Section 4 measures every model.
> 6. CPU by default. A `device` argument is allowed; `score` returns a CPU tensor.
>
> - [ ] Replace the rules above with your own explanation.

For each method, its owner writes a short explanation above the code: the idea, why it is in
the comparison, and its merits and limits.

### 5.1 PopRec (Frank)

> **TODO (Frank)**
> - [x] PopRec: every item's score is how often it appears in `train_seqs`; the same scores
>   for every user.

In [ ]:
# refactored the interface to match the other models, so evaluation and the demo can use
# it directly
class PopRec:
    def __init__(self, n_users, n_items):
        self.n_users = n_users
        self.n_items = n_items
        self.counts = torch.zeros(n_items)

    def train_one_epoch(self, train_seqs):
        items = torch.tensor(
            [item for seq in train_seqs for item in seq], dtype=torch.long
        )
        # Recount each time so repeated epochs do not inflate popularity.
        # minlength keeps zero scores for items absent from training.
        self.counts = torch.bincount(items, minlength=self.n_items).float()
        return 0.0

    # Every user gets the same popularity scores without copying the counts.
    def score(self, users, histories):
        return self.counts.expand(len(users), -1)


pop_rec = PopRec(n_users, n_items)
pop_rec.train_one_epoch(train_seqs)

### 5.2 BPR (Kyle); FMC and FPMC if time allows

BPR [3] learns a vector for each user and item. Their dot product, `P[u] · Q[i]`, is the
item's score for that user. We include it as a personalized baseline: unlike PopRec it can
learn different tastes, but it ignores the order of interactions, unlike SASRec.

Each epoch shuffles the training interactions and pairs each positive item with one uniformly
sampled item absent from that user's **training** history. Validation and test interactions
are not consulted. We minimize `-logsigmoid(positive_score - negative_score)` plus `l2` times
the squared norms of the three sampled vectors, averaged over the batch. `logsigmoid` avoids
numerical overflow when the score difference is large.

We choose Adam with learning rate 0.001 for adaptive updates, batches of 128 to limit memory,
and small normal initial weights (mean 0, standard deviation 0.01) to start with small scores
and distinct vectors. These are our implementation choices; the original BPR paper uses
bootstrap sampling and stochastic gradient descent. Our epoch visits each training interaction
once, including repeats kept by the data loader. Settings are constructor arguments so they
can be inspected and changed. The model uses the shared `train_one_epoch` and `score` interface.

> **Remaining (Kyle)**
> - [ ] Once Section 6's shared training loop is ready, tune `d` in {10, 50} and `l2` in
>   {0.001, 0.01} on sampled validation NDCG@10; then evaluate the chosen settings with
>   seeds 1, 2 and 3 under both protocols and save the results.
> - [ ] If BPR loses to PopRec, confirm that tuning ran before interpreting the result.
> - [ ] Optional after the core results: FMC and FPMC with the same loss, grid and seeds.


In [ ]:
class BPR(torch.nn.Module):
    def __init__(
        self,
        n_users,
        n_items,
        d=50,
        l2=0.001,
        learning_rate=0.001,
        batch_size=128,
        init_std=0.01,
    ):
        super().__init__()
        if batch_size < 1:
            raise ValueError("batch_size is <0")
        self.n_users = n_users
        self.n_items = n_items
        self.l2 = l2
        self.batch_size = batch_size
        self.user_emb = torch.nn.Embedding(n_users, d)
        self.item_emb = torch.nn.Embedding(n_items, d)
        torch.nn.init.normal_(self.user_emb.weight, mean=0.0, std=init_std)
        torch.nn.init.normal_(self.item_emb.weight, mean=0.0, std=init_std)
        self.optimizer = torch.optim.Adam(self.parameters(), lr=learning_rate)

    def train_one_epoch(self, train_seqs):
        examples = [
            (user, item)
            for user, sequence in enumerate(train_seqs)
            for item in sequence
        ]
        random.shuffle(examples)
        self.train()
        total_loss = 0.0
        for start in range(0, len(examples), self.batch_size):
            batch = examples[start : start + self.batch_size]
            users = torch.tensor([user for user, _ in batch], dtype=torch.long)
            positives = torch.tensor([item for _, item in batch], dtype=torch.long)
            negatives = []
            for user, _ in batch:
                # Reject training positives; never inspect held-out interactions.
                item = random.randrange(self.n_items)
                seen = [set(sequence) for sequence in train_seqs]
                while item in seen[user]:
                    item = random.randrange(self.n_items)
                negatives.append(item)
            negatives = torch.tensor(negatives, dtype=torch.long)
            user_vectors = self.user_emb(users)
            positive_vectors = self.item_emb(positives)
            negative_vectors = self.item_emb(negatives)
            difference = (user_vectors * 
                          (positive_vectors - negative_vectors)).sum(dim=1)
            
            ranking_loss = -torch.nn.functional.logsigmoid(difference).mean()
            penalty = (
                user_vectors.square().sum(dim=1)
                + positive_vectors.square().sum(dim=1)
                + negative_vectors.square().sum(dim=1)
            ).mean()
            loss = ranking_loss + self.l2 * penalty
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()
            # Weight the final, possibly smaller batch correctly in the epoch average.
            total_loss += loss.item() * len(batch)
        return total_loss / len(examples)

    @torch.no_grad()
    def score(self, users, histories):
        # BPR uses user identity, the shared evaluator handles history masking.
        users = torch.tensor(list(users), dtype=torch.long)
        return (self.user_emb(users) @ self.item_emb.weight.T).cpu()

### 5.3 SASRec (Alina)

> **TODO (Alina)**
> - [ ] The model with the paper's settings (Sections III and IV-C): 2 self-attention blocks,
>   1 head, learned positional embeddings, item embeddings shared between input and output,
>   at most `MAX_LEN = 50` items, dropout 0.5, Adam with learning rate 0.001, batch size 128,
>   binary cross-entropy with one sampled negative per position.
> - [ ] The explanation: how self-attention reads a sequence, why it is causal, merits and
>   limits compared with PopRec and BPR.
> - [ ] Send the method paragraph to Kyle for Section 1.

## 6. Training and tuning (Alina)

> **TODO** The team agreed on the rules below, so that every trained model is tuned in the
> same way and the baselines get as much tuning as SASRec. Alina builds `train_model` by
> following them, replaces them with her own explanation and presents this part. Kyle follows
> the same rules for his models.
>
> | Model | Tuned on validation | Grid |
> |---|---|---|
> | BPR (FMC, FPMC if time allows) | `d`, `l2` | `d` ∈ {10, 50}; `l2` ∈ {0.001, 0.01}: 4 points |
> | SASRec | `d` | {10, 50}; everything else at the paper's values |
> | PopRec | nothing | — |
>
> The paper tries `d` ∈ {10, 20, 30, 40, 50} and `l2` ∈ {0.0001, …, 1}, and stops after 20
> epochs without improvement (Section IV-B). We keep points inside those ranges but use fewer
> of them, and stop after `PATIENCE = 10`, so that all training fits on our laptops; this goes
> into the limitations (Section 11).
>
> 1. Train every grid point with seed 1 and measure it on validation only.
> 2. Pick the point with the highest sampled validation NDCG@10. If two points tie to four
>    decimals, take the smaller `d`, then the smaller `l2`.
> 3. Train the chosen point with seeds 1, 2 and 3 and measure it on test. **Test is used only
>    here, once.**
>
> Early stopping, for every trained model: after every epoch, measure sampled validation
> NDCG@10; stop after `PATIENCE` epochs without improvement, or at `MAX_EPOCHS`; keep the
> model of the best epoch. Full ranking is reported, never used to choose anything.
>
> - [ ] **Alina:** `train_model(ModelClass, settings, seed)`: calls `set_seed`, creates the
>   model, trains with the early stopping above, and returns the best model with the number of
>   epochs. Kyle uses the same function for his models.
> - [ ] **Everyone with a trained model:** the grid search and the 3 test seeds above, saved to
>   the CSV files of Section 7, with your full grid table shown (one row per grid point), so the
>   reader sees the baselines were tuned too, not only SASRec. For seed 1 also save every
>   user's rank under both protocols in `results/ranks.csv.gz` (Section 7).
> - [ ] **Alina:** save the chosen SASRec (seed 1) with `torch.save(model.state_dict(), ...)`
>   as `results/sasrec_demo.pt`, for the demo (Section 10).
> - [ ] **Frank:** PopRec on test (no tuning), with its ranks.

## 7. Results (Frank)

> **TODO** The team agreed on the file format and names below, so that one table can read
> every member's results. Frank replaces them with his own explanation and presents this part.
>
> Every training run adds rows to small CSV files in `results/`, committed to git. With
> `RUN_EXPERIMENTS = False`, Sections 6 to 9 read these files instead of training.
>
> | File | One row per | Columns |
> |---|---|---|
> | `results/tuning.csv` | grid point | `method, d, l2, valid_ndcg10, epochs` |
> | `results/test.csv` | method, seed, protocol | `method, seed, protocol, hit10, ndcg10, epochs` |
> | `results/ranks.csv.gz` | method, protocol, user (seed 1 only) | `method, protocol, user, rank` |
>
> `protocol` is `sampled` or `full`. `l2` is empty for models without it. Method names:
> `poprec`, `bpr`, `sasrec`, and if time allows `fmc`, `fpmc`; the names of the ablations are
> in Section 9.
>
> - [ ] **Frank:** before comparing anything, check that every method was measured on the same
>   users with the same candidate lists: in `results/ranks.csv.gz` every method has exactly one
>   rank per user and protocol (teacher's feedback).
> - [ ] **Frank:** the main table, like the paper's Table III: one row per method, Hit@10 and
>   NDCG@10, mean ± std over the 3 seeds, sampled and full ranking.
> - [ ] **Kyle:** answer RQ2: does SASRec still rank above the non-neural baselines? Compare
>   the order of the methods with the paper's order (Section 1), not the exact numbers.
> - [ ] **Matthew:** figures for the table.

## 8. Sampled evaluation vs full ranking: our main question

> **TODO**
> - [ ] **Frank:** RQ1: do the sampled protocol and full ranking put the methods in the same
>   order? Show both orders side by side and explain any change. Krichene and Rendle [5] show
>   that sampled metrics can change the order of methods; say whether we see the same.
> - [ ] **Matthew:** results by the length of the test input history, as the teacher
>   suggested: groups of 4–5, 6–10 and more than 10 items (5-core data should give every user
>   at least 5 items, so at least 4 in the test history; check it). From
>   `results/ranks.csv.gz`, for PopRec, BPR and SASRec under both protocols: Hit@10 and
>   NDCG@10 per group, with the number of users per group, in one figure. Does the gap between
>   sampled and full ranking change with the history length?

## 9. Ablations: which parts of SASRec matter? (Alina, if time allows)

> **TODO (Alina)** Only after the core results are in. The three ablations of the proposal,
> with the names for the CSV files of Section 7:
>
> | Variant | Setting | Method name |
> |---|---|---|
> | no positional embedding | `use_pe=False` | `sasrec_nope` |
> | separate input and output item embeddings | `share_item_emb=False` | `sasrec_unshared` |
> | 1 or 3 blocks (2 is the default) | `n_blocks=1`, `3` | `sasrec_blocks1`, `sasrec_blocks3` |
>
> - [ ] The switches `use_pe`, `share_item_emb` and `n_blocks` in the SASRec class (Section 5.3).
> - [ ] Every variant with the 3 test seeds, and answer RQ3.

## 10. Demo (Matthew)

> **TODO (Matthew)** The course requires a recorded demo video that shows the recommender
> working. It is played in class and paused to comment. Walking through the code is not a demo.
> The whole presentation, video included, must fit in 20 minutes, so aim for about 3 minutes of
> video. The video must show the version we hand in on November 30.
>
> First build the demo here:
>
> - [ ] Product titles: the 5-core files have only `parent_asin`. Download the metadata file of
>   `CATEGORY` from the Amazon Reviews 2023 page, keep only `parent_asin` and `title` of our
>   items, save that as `data/<CATEGORY>/titles.csv.gz` and commit it (not the full metadata
>   file).
> - [ ] `show_user(u, history)`: loads SASRec from `results/sasrec_demo.pt` (Alina saves it,
>   Section 6) and builds PopRec (Section 5.1); prints the history as titles, oldest first,
>   then the top 10 of SASRec and of PopRec from `recommend` (Section 4), and marks the user's
>   real next item (`test_answers[u]`) if it is in a list.
> - [ ] Pick the users before recording: two users with clearly different histories where
>   SASRec finds the real next item, and one user where it does not. Say in the video that
>   they were picked.
>
> Then record it. The scenes, in this order; each one is a place to pause and comment:
>
> 1. One sentence: "Given what a user reviewed so far, we recommend the next product."
> 2. User A: the history, then SASRec's top 10 with the real next item marked.
> 3. The same user with PopRec: the same list for every user. This is the baseline.
> 4. User B, with different taste: SASRec's list changes; PopRec's does not.
> 5. The user where SASRec misses: an honest example of its limits.
>
> - [ ] Record only outputs (run the cells and scroll to the results; do not explain code), in
>   1080p (macOS: QuickTime, File → New Screen Recording; Windows: Win + Alt + R), with a short
>   title line above each scene. A voice-over is optional, because we comment live in class.
>   Save it as `demo.mp4` and upload it to Moodle with the hand-in; do not commit it to git.
> - [ ] Agree with the team on who comments at which pause.

## 11. Discussion

> **TODO**
> - [ ] **Everyone:** observations, limitations and future work for your own part. Already
>   known: our tuning grid is smaller than the paper's and we stop training earlier
>   (Section 6), and whatever we did not reach under "if time allows". One idea for future
>   work: the official date-based split (`timestamp_w_his`) is closer to real use than
>   leave-one-out.
> - [ ] **Matthew:** first draft of the conclusion: the answers to RQ1 to RQ3 (RQ1 first, it is
>   our main question), which conclusions agree with the paper, which do not, and possible
>   reasons. Everyone reviews it.
> - [ ] **Everyone:** cite every paper, dataset and library you use in your section as [n]
>   (Section 12); write your row of the contribution statement and the AI note in Section 0.